In [1]:
import os,sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from pathlib import Path 

# Load Datasets
url_orders = "https://mydataaijournal.com/public/andy-data/olist-data/raw_data/olist_orders_dataset.csv"

df_orders_raw = pd.read_csv(url_orders)
print(f"Successfully loaded {len(df_orders_raw):,} orders!")

df_orders = df_orders_raw.copy()

# Get the directory one level above CWD
PROJECT_ROOT = Path.cwd().resolve().parent
#print(f"Project Root: {PROJECT_ROOT}")

LIB_PATH = PROJECT_ROOT / "utility"
sys.path.append(str(LIB_PATH))

import clean_lib  # type:ignore


Successfully loaded 99,441 orders!


In [2]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [3]:
print(df_orders.dtypes)


order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


In [4]:
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
rename_cols = {
    "order_purchase_timestamp":"order_purchase_at",
    "order_delivered_carrier_date":"order_delivered_carrier_at",
    "order_delivered_customer_date":"order_delivered_customer_at",
    "order_estimated_delivery_date":"order_estimated_delivery_at"
}

df_orders = df_orders.rename(columns=rename_cols)
df_orders.head(3)

,order_id,customer_id,order_status,order_purchase_at,order_approved_at,order_delivered_carrier_at,order_delivered_customer_at,order_estimated_delivery_at
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [6]:
df_orders.order_id.sample(3)

79778    71ca3bab201007339dfb592df2d185d4
67906    3220cd63fe43f4d8ec90a2f1a2925714
25232    ddf02da047a1c300e78346fe13f7e862
Name: order_id, dtype: object

<div style="width:90%;margin:auto;">

### order_id

- The **`order_id`** is a unique primary key representing a single purchase transaction. In the Olist dataset, it is anonymized as a 32-character hexadecimal MD5 hash key (e.g., `07677527dea7bbe02d91875f7764c6cd61511`).

- In **SQL (PostgreSQL / Snowflake)**, cast it explicitly to **`VARCHAR(32)`** to enforce fixed-length schema validation.

```sql
SELECT order_id::VARCHAR(32) AS order_id FROM raw_source;

In [7]:
df_orders["order_id"] = df_orders["order_id"].astype("string")

df_orders.order_id.dtype

string[python]

<div style="width:90%;margin:auto;">

### customer_id

- The **`customer_id`** is a unique 32-character hexadecimal MD5 hash key assigned to a specific order transaction. In Olist, a new `customer_id` is generated for every purchase to link the order directly to the customer's profile.

- In **SQL (PostgreSQL / Snowflake)**, it is explicitly cast as a **`VARCHAR(32)`** to enforce fixed-length schema validation.

```sql
SELECT customer_id::VARCHAR(32) AS customer_id FROM raw_source;

In [8]:
df_orders.customer_id.sample(3)

89054    91674630c3abf8b348512504008eb8e0
61025    771460028f9bb4394935112dac04a6a4
16028    2edb1f5440fd7a15efecea507255273b
Name: customer_id, dtype: object

In [9]:
df_orders["customer_id"] = df_orders.customer_id.astype("string")

df_orders.customer_id.dtype

string[python]

### order_status

In [10]:
display(df_orders.order_status.value_counts(dropna=False))

display(df_orders.order_status.value_counts(dropna=False,normalize=True).round(2))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

order_status
delivered      0.97
shipped        0.01
canceled       0.01
unavailable    0.01
invoiced       0.00
processing     0.00
created        0.00
approved       0.00
Name: proportion, dtype: float64

<div style="width:90%;margin:auto;">

### order_status Analysis

- **Primary State:** **97% of orders** (`96,478`) are in the final `delivered` state, showing that the dataset primarily consists of completed e-commerce transactions.
- **Fulfillment Pipeline:** `order_status` is a dynamic categorical column representing the real-time stage of an order in Olist's fulfillment pipeline:
  1. `created` $\rightarrow$ Order initialized by customer.
  2. `approved` $\rightarrow$ Payment confirmed.
  3. `invoiced` $\rightarrow$ Invoice issued by seller.
  4. `processing` $\rightarrow$ Order being packed/prepared.
  5. `shipped` $\rightarrow$ Handed over to logistics carrier.
  6. `delivered` $\rightarrow$ Received by customer (terminal success state).
  7. `canceled` / `unavailable` $\rightarrow$ Exception states (terminal failure states).

</div>

In [11]:
df_orders.order_status.unique()

array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
       'canceled', 'created', 'approved'], dtype=object)

In [12]:
df_orders["order_status"] = df_orders["order_status"].astype("category")
df_orders.order_status.dtype

CategoricalDtype(categories=['approved', 'canceled', 'created', 'delivered', 'invoiced',
                  'processing', 'shipped', 'unavailable'],
, ordered=False, categories_dtype=object)

<div style="width:90%;margin:auto;">

### order_status Data Type Mapping

- **Classification:** **Nominal Categorical** (`ordered=False`)
- **Reasoning:** Represents distinct operational states with no numeric ranking. 

#### Snowflake / Postgresql

```sql
SELECT order_status::VARCHAR(50) AS order_status FROM raw_orders;
```

</div>


<div style="width:85%;margin:auto;">

### Olist Order Lifecycle Timestamps

These five columns record the critical milestones of an order as it moves through Olist's fulfillment pipeline:

1. **`order_purchase_at`** (100% complete — `99,441` non-null)
* **Meaning:** The exact timestamp when the customer placed the order on the e-commerce marketplace.
* **Pipeline Stage:** Order Creation (`created`).


2. **`order_approved_at`** (`99,281` non-null)
* **Meaning:** The timestamp when payment was approved by the payment gateway.
* **Pipeline Stage:** Payment Approval (`approved`).
* *Note:* The 160 missing values represent orders where payment failed or was pending when canceled.


3. **`order_delivered_carrier_at`** (`97,658` non-null)
* **Meaning:** The timestamp when the seller handed the package over to the courier/logistics partner.
* **Pipeline Stage:** Handover to Freight Carrier (`shipped`).
* *Note:* Missing values represent orders still being processed by the seller, or canceled before shipment.


4. **`order_delivered_customer_at`** (`96,476` non-null)
* **Meaning:** The actual timestamp when the package was delivered to the customer's doorstep.
* **Pipeline Stage:** Successful Delivery (`delivered`).
* *Note:* Matches closely with the 96,478 `delivered` count in `order_status`.


5. **`order_estimated_delivery_at`** (100% complete — `99,441` non-null)
* **Meaning:** The promised delivery date shown to the customer at checkout.
* **Pipeline Stage:** SLA Benchmark.
* *Note:* Critical for calculating delivery performance (comparing actual `order_delivered_customer_at` vs. estimated).



---

### Data Type Conversions & Mapping

Currently, all 5 columns are stored as legacy **`object`** text strings. They should be parsed into true datetime types:



Cast to **`TIMESTAMP`** in your staging SQL model:

```sql
SELECT
    order_purchase_at::TIMESTAMP          AS order_purchase_at,
    order_approved_at::TIMESTAMP          AS order_approved_at,
    order_delivered_carrier_at::TIMESTAMP AS order_delivered_carrier_at,
    order_delivered_customer_at::TIMESTAMP AS order_delivered_customer_at,
    order_estimated_delivery_at::TIMESTAMP AS order_estimated_delivery_at
FROM raw_orders;

```

---

</div>

In [13]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   order_id                     99441 non-null  string  
 1   customer_id                  99441 non-null  string  
 2   order_status                 99441 non-null  category
 3   order_purchase_at            99441 non-null  object  
 4   order_approved_at            99281 non-null  object  
 5   order_delivered_carrier_at   97658 non-null  object  
 6   order_delivered_customer_at  96476 non-null  object  
 7   order_estimated_delivery_at  99441 non-null  object  
dtypes: category(1), object(5), string(2)
memory usage: 5.4+ MB


In [14]:
df_orders_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [15]:
timestamp_cols = [
    "order_purchase_at",
    "order_approved_at",
    "order_delivered_carrier_at",
    "order_delivered_customer_at",
    "order_estimated_delivery_at"
]

df_orders[timestamp_cols] = df_orders[timestamp_cols].apply(pd.to_datetime)

df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   order_id                     99441 non-null  string        
 1   customer_id                  99441 non-null  string        
 2   order_status                 99441 non-null  category      
 3   order_purchase_at            99441 non-null  datetime64[ns]
 4   order_approved_at            99281 non-null  datetime64[ns]
 5   order_delivered_carrier_at   97658 non-null  datetime64[ns]
 6   order_delivered_customer_at  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_at  99441 non-null  datetime64[ns]
dtypes: category(1), datetime64[ns](5), string(2)
memory usage: 5.4 MB


In [16]:
clean_lib.null_data(df_orders)

,order_id,customer_id,order_status,order_purchase_at,order_approved_at,order_delivered_carrier_at,order_delivered_customer_at,order_estimated_delivery_at
Total Null,0,0,0,0,160,1783,2965,0
Percent Null,0.0,0.0,0.0,0.0,0.2,1.8,3.0,0.0
Types,string,string,category,datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns],datetime64[ns]
